## Importing Required Libraries

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from FDApy import IrregularFunctionalData
from FDApy.representation import (
    DenseArgvals,
    IrregularArgvals,
    IrregularValues
)
from FDApy.preprocessing import UFPCA

## Define Project Folder Paths

In [2]:
# Root folder of the ICU Clustering project
ROOT_DIR = Path.cwd().parent

# Folder containing the raw eICU CSV files
DATA_DIR = ROOT_DIR / "data"

# Folder where processed datasets from this notebook will be saved
RESULTS_DIR = (
    ROOT_DIR
    / "results"
    / "1.dataset_creation"
)

# Create the results folder if it does not already exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify that the paths are pointing to the expected locations
print("Root directory:", ROOT_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Root directory: C:\Users\samsa\Documents\ICU Clustering
Data directory: C:\Users\samsa\Documents\ICU Clustering\data
Results directory: C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation


#### Observation :
The project paths are defined so the notebook can consistently read raw eICU data and save processed dataset outputs in the correct results folder.

### Load the Patient Table

In [3]:
# Load only the patient-table columns needed for cohort selection
# and later demographic/admission feature construction
patient_columns = [
    "patientunitstayid",   # Unique ICU stay identifier
    "uniquepid",           # Unique patient identifier
    "unitvisitnumber",     # ICU visit number during the hospitalization
    "hospitaladmitoffset", # Time offset of hospital admission relative to ICU admission
    "apacheadmissiondx",   # APACHE admission diagnosis used to identify sepsis
    "age",                 # Patient age
    "gender",              # Patient gender
    "ethnicity",           # Patient ethnicity
    "unittype",            # ICU unit type
    "admissionheight",     # Height recorded at ICU admission
    "admissionweight"      # Weight recorded at ICU admission
]

# Read the selected columns from the eICU patient table
patient = pd.read_csv(
    DATA_DIR / "patient.csv",
    usecols=patient_columns
)

# Show the starting size of the available eICU population
print("Dataset shape:", patient.shape)
print("Total ICU stay records:", len(patient))
print("Unique ICU stays:", patient["patientunitstayid"].nunique())
print("Unique patients:", patient["uniquepid"].nunique())

Dataset shape: (200859, 11)
Total ICU stay records: 200859
Unique ICU stays: 200859
Unique patients: 139367


#### Observation :
The starting eICU patient table contains `200,859` ICU stay records and each `patientunitstayid` is unique, confirming one row per ICU stay. These stays belong to `139,367` unique patients which means some patients have more than one ICU stay

## Identify Sepsis ICU Stay

In [4]:
# Record the dataset size before applying the sepsis filter
before_shape = patient.shape
before_stays = len(patient)
before_patients = patient["uniquepid"].nunique()

# Identify ICU stays where the APACHE admission diagnosis contains "sepsis"
# case=False ignores uppercase/lowercase differences
# na=False treats missing diagnosis values as non-sepsis
sepsis_mask = patient["apacheadmissiondx"].str.contains(
    "sepsis",
    case=False,
    na=False
)

# Keep only ICU stays identified as sepsis
sepsis = patient.loc[sepsis_mask].copy()

# Record the dataset size after filtering
after_shape = sepsis.shape
after_stays = len(sepsis)
after_patients = sepsis["uniquepid"].nunique()

# Calculate how many ICU stays and unique patients were excluded
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before sepsis filtering:", before_shape)

print("\nSepsis admission diagnosis categories:")
print(
    sepsis["apacheadmissiondx"]
    .value_counts()
)

print("\nBefore sepsis filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved / not identified as sepsis")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter sepsis filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nDataset shape after sepsis filtering:", after_shape)

Dataset shape before sepsis filtering: (200859, 11)

Sepsis admission diagnosis categories:
apacheadmissiondx
Sepsis, pulmonary                        8862
Sepsis, renal/UTI (including bladder)    5273
Sepsis, GI                               2881
Sepsis, unknown                          2602
Sepsis, cutaneous/soft tissue            1933
Sepsis, other                            1510
Sepsis, gynecologic                        75
Name: count, dtype: int64

Before sepsis filtering
ICU stays: 200859
Unique patients: 139367

Removed / not identified as sepsis
ICU stays: 177723
Unique patients: 119236

After sepsis filtering
ICU stays: 23136
Unique patients: 20131

Dataset shape after sepsis filtering: (23136, 11)


#### Observation :

Before sepsis filtering, the dataset contained `200,859` ICU stays from `139,367` unique patients.

The filter excluded `177,723` ICU stays and `119,236` unique patients that were not identified as sepsis.

After filtering, `23,136` sepsis ICU stays from `20,131` unique patients remained, meaning some patients had multiple sepsis ICU stays.

The largest sepsis category was pulmonary sepsis (`8,862`), followed by renal/UTI sepsis (`5,273`).


## Keep One Sepsis ICU Stay per Patient

In [5]:
# Record cohort information before removing repeated sepsis ICU stays
before_shape = sepsis.shape
before_stays = len(sepsis)
before_patients = sepsis["uniquepid"].nunique()

# Sort ICU stays so the earliest sepsis ICU visit for each patient appears first
sepsis_sorted = sepsis.sort_values(
    by=["uniquepid", "unitvisitnumber", "hospitaladmitoffset"]
)

# Keep only one sepsis ICU stay for each unique patient
sepsis_first = (
    sepsis_sorted
    .drop_duplicates(subset="uniquepid", keep="first")
    .copy()
)

# Record cohort information after removing repeated stays
after_shape = sepsis_first.shape
after_stays = len(sepsis_first)
after_patients = sepsis_first["uniquepid"].nunique()

# Calculate the number of ICU stays and patients removed
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before selecting one ICU stay per patient:", before_shape)

print("\nSepsis admission diagnosis categories after selection:")
print(
    sepsis_first["apacheadmissiondx"]
    .value_counts()
)

print("\nBefore selecting one sepsis ICU stay per patient")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved repeated sepsis ICU stays")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter selecting one sepsis ICU stay per patient")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nDataset shape after selection:", after_shape)

Dataset shape before selecting one ICU stay per patient: (23136, 11)

Sepsis admission diagnosis categories after selection:
apacheadmissiondx
Sepsis, pulmonary                        7672
Sepsis, renal/UTI (including bladder)    4550
Sepsis, GI                               2545
Sepsis, unknown                          2313
Sepsis, cutaneous/soft tissue            1693
Sepsis, other                            1286
Sepsis, gynecologic                        72
Name: count, dtype: int64

Before selecting one sepsis ICU stay per patient
ICU stays: 23136
Unique patients: 20131

Removed repeated sepsis ICU stays
ICU stays: 3005
Unique patients: 0

After selecting one sepsis ICU stay per patient
ICU stays: 20131
Unique patients: 20131

Dataset shape after selection: (20131, 11)


#### Observation :

Before selecting one ICU stay per patient, the dataset contained `23,136` sepsis ICU stays from `20,131` unique patients.

A total of `3,005` repeated sepsis ICU stays were removed, while no unique patients were lost.

After this step, `20,131` ICU stays from `20,131` unique patients remained, meaning each patient is represented by only one sepsis ICU stay.

Pulmonary sepsis remained the largest category with `7,672` patients, followed by renal/UTI sepsis with `4,550`.

## Clean Age and Keep Adult Patients

In [6]:
# Record cohort information before age preprocessing
before_shape = sepsis_first.shape
before_stays = len(sepsis_first)
before_patients = sepsis_first["uniquepid"].nunique()

# eICU records patients older than 89 as "> 89"
# Convert this category to 90 so age can be treated numerically
over_89_count = (sepsis_first["age"] == "> 89").sum()

sepsis_first["age"] = (
    sepsis_first["age"]
    .replace("> 89", 90)
)

# Convert age to numeric
# Any invalid values would become NaN
sepsis_first["age"] = pd.to_numeric(
    sepsis_first["age"],
    errors="coerce"
)

# Keep only adult patients aged 18 years or older
adult_sepsis = sepsis_first[
    sepsis_first["age"] >= 18
].copy()

# Record cohort information after adult filtering
after_shape = adult_sepsis.shape
after_stays = len(adult_sepsis)
after_patients = adult_sepsis["uniquepid"].nunique()

# Calculate removals
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before age filtering:", before_shape)

print("\nSepsis admission diagnosis categories after age filtering:")
print(
    adult_sepsis["apacheadmissiondx"]
    .value_counts()
)

print("\nPatients recorded as > 89:", over_89_count)

print("\nBefore age filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved patients younger than 18")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter age filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nDataset shape after age filtering:", after_shape)
print("Minimum age:", adult_sepsis["age"].min())
print("Maximum age:", adult_sepsis["age"].max())

Dataset shape before age filtering: (20131, 11)

Sepsis admission diagnosis categories after age filtering:
apacheadmissiondx
Sepsis, pulmonary                        7670
Sepsis, renal/UTI (including bladder)    4545
Sepsis, GI                               2544
Sepsis, unknown                          2313
Sepsis, cutaneous/soft tissue            1691
Sepsis, other                            1284
Sepsis, gynecologic                        72
Name: count, dtype: int64

Patients recorded as > 89: 1072

Before age filtering
ICU stays: 20131
Unique patients: 20131

Removed patients younger than 18
ICU stays: 12
Unique patients: 12

After age filtering
ICU stays: 20119
Unique patients: 20119

Dataset shape after age filtering: (20119, 11)
Minimum age: 18
Maximum age: 90


#### Observation :
Before age filtering, the dataset contained `20,131` ICU stays from `20,131` unique patients.

A total of `12` patients younger than `18` were removed.

After filtering, `20,119` adult sepsis patients remained, with ages ranging from `18` to `90`.

The sepsis diagnosis distribution changed only slightly, so age filtering had minimal effect on the cohort composition.

## Remove Cardiothoracic Surgical ICU Patients

In [7]:
# Record cohort information before removing CSICU patients
before_shape = adult_sepsis.shape
before_stays = len(adult_sepsis)
before_patients = adult_sepsis["uniquepid"].nunique()

# ICU type distribution before filtering
before_unit_distribution = (
    adult_sepsis["unittype"]
    .value_counts()
)

# Identify patients admitted to the Cardiothoracic Surgical ICU
csicu_mask = adult_sepsis["unittype"].eq("CSICU")

# Remove CSICU patients
sepsis_no_csicu = adult_sepsis.loc[~csicu_mask].copy()

# Record cohort information after filtering
after_shape = sepsis_no_csicu.shape
after_stays = len(sepsis_no_csicu)
after_patients = sepsis_no_csicu["uniquepid"].nunique()

# ICU type distribution after filtering
after_unit_distribution = (
    sepsis_no_csicu["unittype"]
    .value_counts()
)

# Calculate removals
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before CSICU filtering:", before_shape)

print("\nICU type distribution before CSICU filtering:")
print(before_unit_distribution)

print("\nBefore CSICU filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved CSICU patients")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter CSICU filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nICU type distribution after CSICU filtering:")
print(after_unit_distribution)

print("\nDataset shape after CSICU filtering:", after_shape)

Dataset shape before CSICU filtering: (20119, 11)

ICU type distribution before CSICU filtering:
unittype
Med-Surg ICU    13532
MICU             2631
Cardiac ICU      1219
CCU-CTICU        1157
SICU              742
Neuro ICU         375
CSICU             315
CTICU             148
Name: count, dtype: int64

Before CSICU filtering
ICU stays: 20119
Unique patients: 20119

Removed CSICU patients
ICU stays: 315
Unique patients: 315

After CSICU filtering
ICU stays: 19804
Unique patients: 19804

ICU type distribution after CSICU filtering:
unittype
Med-Surg ICU    13532
MICU             2631
Cardiac ICU      1219
CCU-CTICU        1157
SICU              742
Neuro ICU         375
CTICU             148
Name: count, dtype: int64

Dataset shape after CSICU filtering: (19804, 11)


#### Observation :

Before CSICU filtering, the dataset contained `20,119` ICU stays from `20,119` unique patients.

The ICU type distribution included `315` patients in the CSICU.

A total of `315` ICU stays and `315` unique patients were removed.

After filtering, `19,804` ICU stays from `19,804` unique patients remained.

The remaining cohort is mostly composed of Med-Surg ICU patients (`13,532`), followed by MICU (`2,631`) and Cardiac ICU (`1,219`).

## Remove Patients with Missing Demographic Information

In [8]:
# Record cohort information before demographic filtering
before_shape = sepsis_no_csicu.shape
before_stays = len(sepsis_no_csicu)
before_patients = sepsis_no_csicu["uniquepid"].nunique()

# Demographic fields required for the analysis
required_demographics = [
    "age",
    "gender",
    "ethnicity",
    "unittype"
]

# Check missing demographic values before filtering
missing_before = (
    sepsis_no_csicu[required_demographics]
    .isna()
    .sum()
)

# Store category distributions before filtering
gender_before = sepsis_no_csicu["gender"].value_counts(dropna=False)
ethnicity_before = sepsis_no_csicu["ethnicity"].value_counts(dropna=False)

# Remove patients missing any required demographic field
sepsis_complete_demo = (
    sepsis_no_csicu
    .dropna(subset=required_demographics)
    .copy()
)

# Record cohort information after demographic filtering
after_shape = sepsis_complete_demo.shape
after_stays = len(sepsis_complete_demo)
after_patients = sepsis_complete_demo["uniquepid"].nunique()

# Calculate removals
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients

# Check missing values after filtering
missing_after = (
    sepsis_complete_demo[required_demographics]
    .isna()
    .sum()
)

# Store category distributions after filtering
gender_after = sepsis_complete_demo["gender"].value_counts(dropna=False)
ethnicity_after = sepsis_complete_demo["ethnicity"].value_counts(dropna=False)


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before demographic filtering:", before_shape)

print("\nMissing demographic values before filtering:")
print(missing_before)

print("\nGender distribution before filtering:")
print(gender_before)

print("\nEthnicity distribution before filtering:")
print(ethnicity_before)

print("\nBefore demographic filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved because of missing demographic information")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter demographic filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nMissing demographic values after filtering:")
print(missing_after)

print("\nGender distribution after filtering:")
print(gender_after)

print("\nEthnicity distribution after filtering:")
print(ethnicity_after)

print("\nDataset shape after demographic filtering:", after_shape)

Dataset shape before demographic filtering: (19804, 11)

Missing demographic values before filtering:
age            0
gender         4
ethnicity    187
unittype       0
dtype: int64

Gender distribution before filtering:
gender
Male       10044
Female      9754
NaN            4
Unknown        2
Name: count, dtype: int64

Ethnicity distribution before filtering:
ethnicity
Caucasian           15400
African American     1912
Other/Unknown         956
Hispanic              784
Asian                 387
NaN                   187
Native American       178
Name: count, dtype: int64

Before demographic filtering
ICU stays: 19804
Unique patients: 19804

Removed because of missing demographic information
ICU stays: 187
Unique patients: 187

After demographic filtering
ICU stays: 19617
Unique patients: 19617

Missing demographic values after filtering:
age          0
gender       0
ethnicity    0
unittype     0
dtype: int64

Gender distribution after filtering:
gender
Male       9940
Female     

#### Observation :
Before demographic filtering, the dataset contained `19,804` ICU stays from `19,804` unique patients.

There were `4` missing gender values and `187` missing ethnicity values. Because some missing values occurred in the same patients, `187` patients were removed in total.

After filtering, `19,617` ICU stays from `19,617` unique patients remained, with no missing values in age, gender, ethnicity, or ICU type.

Unknown gender and Other/Unknown ethnicity were retained because they are recorded categories rather than missing values.

## Load the Laboratory Table

In [9]:
# Load only the laboratory columns required for feature construction
lab_columns = [
    "patientunitstayid",   # ICU stay identifier used to link with the patient cohort
    "labresultoffset",     # Time of the laboratory measurement relative to ICU admission
    "labname",             # Name of the laboratory test
    "labresult"            # Recorded laboratory value
]

# Read the eICU laboratory table
lab = pd.read_csv(
    DATA_DIR / "lab.csv",
    usecols=lab_columns
)

# Display the size and coverage of the raw laboratory table
print("Laboratory dataset shape:", lab.shape)
print("Total laboratory records:", len(lab))
print("Unique ICU stays with laboratory records:", lab["patientunitstayid"].nunique())

print("\nNumber of unique laboratory test names:", lab["labname"].nunique())

Laboratory dataset shape: (39132531, 4)
Total laboratory records: 39132531
Unique ICU stays with laboratory records: 195730

Number of unique laboratory test names: 158


#### Observation :

The raw laboratory table contains `39,132,531` laboratory records across `195,730` ICU stays.

There are `158` distinct laboratory test names in the full table.

This table will now be restricted to the selected laboratory variables and to the patients in the current sepsis cohort.

## Select Laboratory Variables and Restrict ti the First 24 hours

In [10]:
# Laboratory variables selected for the clustering analysis
selected_labs = [
    "bicarbonate",
    "lactate",
    "potassium",
    "platelets x 1000",
    "anion gap",
    "chloride",
    "BUN",
    "creatinine",
    "sodium",
    "glucose",
    "WBC x 1000",
    "Hgb"
]

# Record laboratory table information before filtering
before_shape = lab.shape
before_records = len(lab)
before_icu_stays = lab["patientunitstayid"].nunique()

# Keep valid measurements of the 12 selected laboratory variables
selected_lab_all = lab[
    (lab["labname"].isin(selected_labs))
    & (lab["labresult"].notna())
].copy()

# Distribution of selected laboratory measurements before
# restricting to the sepsis cohort and first 24 hours
lab_distribution_before = (
    selected_lab_all["labname"]
    .value_counts()
    .reindex(selected_labs)
)

# ICU stay IDs belonging to the current sepsis cohort
candidate_ids = set(
    sepsis_complete_demo["patientunitstayid"]
)

# Keep selected laboratory measurements:
# 1. belonging to the current cohort
# 2. recorded from ICU admission through the first 24 hours
# 3. having a valid laboratory result
valid_lab_24h = lab[
    (lab["patientunitstayid"].isin(candidate_ids))
    & (lab["labresultoffset"] >= 0)
    & (lab["labresultoffset"] <= 1440)
    & (lab["labname"].isin(selected_labs))
    & (lab["labresult"].notna())
].copy()

# Record information after laboratory extraction
after_shape = valid_lab_24h.shape
after_records = len(valid_lab_24h)
after_icu_stays = valid_lab_24h["patientunitstayid"].nunique()

# Distribution of selected labs in the current cohort during first 24 hours
lab_distribution_after = (
    valid_lab_24h["labname"]
    .value_counts()
    .reindex(selected_labs)
)

# Determine current cohort coverage
cohort_patients = sepsis_complete_demo["patientunitstayid"].nunique()
patients_with_labs = valid_lab_24h["patientunitstayid"].nunique()
patients_without_labs = cohort_patients - patients_with_labs


# ---------------------------------------------------------
# Display laboratory extraction information
# ---------------------------------------------------------

print("Laboratory dataset shape before filtering:", before_shape)

print("\nSelected laboratory measurement distribution before filtering:")
print(lab_distribution_before)

print("\nBefore laboratory filtering")
print("Laboratory records:", before_records)
print("ICU stays with laboratory records:", before_icu_stays)

print("\nAfter restricting to current cohort and first 24 hours")
print("Laboratory records:", after_records)
print("ICU stays with at least one selected lab:", after_icu_stays)

print("\nSelected laboratory measurement distribution after filtering:")
print(lab_distribution_after)

print("\nCurrent sepsis cohort:", cohort_patients)
print("Patients with at least one selected lab:", patients_with_labs)
print("Patients with no selected labs:", patients_without_labs)

print("\nLaboratory dataset shape after filtering:", after_shape)

Laboratory dataset shape before filtering: (39132531, 4)

Selected laboratory measurement distribution before filtering:
labname
bicarbonate         1197567
lactate              204602
potassium           1492987
platelets x 1000    1139619
anion gap           1023147
chloride            1283689
BUN                 1268397
creatinine          1275796
sodium              1393097
glucose             1319246
WBC x 1000          1134339
Hgb                 1298443
Name: count, dtype: int64

Before laboratory filtering
Laboratory records: 39132531
ICU stays with laboratory records: 195730

After restricting to current cohort and first 24 hours
Laboratory records: 338083
ICU stays with at least one selected lab: 18722

Selected laboratory measurement distribution after filtering:
labname
bicarbonate         28002
lactate             22751
potassium           34736
platelets x 1000    23933
anion gap           23865
chloride            30091
BUN                 29808
creatinine          29940

#### Observation :
Before laboratory filtering, the raw lab table contained `39,132,531` records across `195,730` ICU stays.

After restricting to the current sepsis cohort, the selected `12` laboratory variables, and the first `24` hours, `338,083` laboratory records remained across `18,722` ICU stays.

From the current `19,617` sepsis patients, `18,722` had at least one selected laboratory measurement and `895` had none.

Among the selected labs, potassium had the highest number of first 24 hour measurements (`34,736`), while lactate had the fewest (`22,751`).

## Check Laboratory Availability and Keep Patients With All 12 Selected Labs

In [11]:
# Record cohort information before applying laboratory completeness criteria
before_shape = sepsis_complete_demo.shape
before_stays = len(sepsis_complete_demo)
before_patients = sepsis_complete_demo["uniquepid"].nunique()

# Create a patient-by-laboratory availability table
# Each cell contains the number of measurements available for that lab
lab_availability = (
    valid_lab_24h
    .groupby(["patientunitstayid", "labname"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=selected_labs, fill_value=0)
)

# Count how many patients have at least one measurement of each laboratory test
lab_patient_counts_before = (
    (lab_availability > 0)
    .sum()
)

# Identify patients who have at least one measurement
# for every one of the 12 selected laboratory variables
has_all_labs = (lab_availability > 0).all(axis=1)

patients_with_all_labs = lab_availability.index[has_all_labs]

# Keep only patients satisfying the complete laboratory requirement
sepsis_complete_labs = (
    sepsis_complete_demo[
        sepsis_complete_demo["patientunitstayid"].isin(patients_with_all_labs)
    ]
    .copy()
)

# Record cohort information after laboratory completeness filtering
after_shape = sepsis_complete_labs.shape
after_stays = len(sepsis_complete_labs)
after_patients = sepsis_complete_labs["uniquepid"].nunique()

# Calculate removals
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients

# After filtering, every retained patient has all 12 selected labs
lab_patient_counts_after = pd.Series(
    after_patients,
    index=selected_labs,
    name="patients"
)


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before laboratory completeness filtering:", before_shape)

print("\nPatients with each selected laboratory before filtering:")
print(lab_patient_counts_before)

print("\nBefore laboratory completeness filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved because one or more selected labs were missing")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter laboratory completeness filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nPatients with each selected laboratory after filtering:")
print(lab_patient_counts_after)

print("\nDataset shape after laboratory completeness filtering:", after_shape)

Dataset shape before laboratory completeness filtering: (19617, 11)

Patients with each selected laboratory before filtering:
labname
bicarbonate         16781
lactate             11372
potassium           18137
platelets x 1000    17493
anion gap           14396
chloride            18007
BUN                 18048
creatinine          18063
sodium              18125
glucose             17982
WBC x 1000          17634
Hgb                 17699
dtype: int64

Before laboratory completeness filtering
ICU stays: 19617
Unique patients: 19617

Removed because one or more selected labs were missing
ICU stays: 11305
Unique patients: 11305

After laboratory completeness filtering
ICU stays: 8312
Unique patients: 8312

Patients with each selected laboratory after filtering:
bicarbonate         8312
lactate             8312
potassium           8312
platelets x 1000    8312
anion gap           8312
chloride            8312
BUN                 8312
creatinine          8312
sodium              8312
gl

#### Observation :

Before laboratory completeness filtering, the dataset contained `19,617` ICU stays from `19,617` unique patients.

A total of `11,305` patients were removed because they were missing at least one of the `12` selected laboratory variables within the first 24 hours.

After filtering, `8,312` ICU stays from `8,312` unique patients remained.

Lactate had the lowest availability before filtering with `11,372` patients, making it one of the main variables limiting cohort size.

## Load the Vital Signs Table

In [12]:
# Load only the vital-sign columns required for the analysis
vital_columns = [
    "patientunitstayid",   # ICU stay identifier
    "observationoffset",   # Time of measurement relative to ICU admission
    "sao2",                # Oxygen saturation
    "heartrate",           # Heart rate
    "respiration"          # Respiratory rate
]

# Read the selected columns from vitalPeriodic.csv
vitals = pd.read_csv(
    DATA_DIR / "vitalPeriodic.csv",
    usecols=vital_columns
)

# Display the size and coverage of the raw vital-sign table
print("Vital-sign dataset shape:", vitals.shape)
print("Total vital-sign records:", len(vitals))
print("Unique ICU stays with vital-sign records:", vitals["patientunitstayid"].nunique())

print("\nNon-missing vital-sign measurements:")
print(
    vitals[
        ["sao2", "heartrate", "respiration"]
    ]
    .notna()
    .sum()
)

Vital-sign dataset shape: (146671642, 5)
Total vital-sign records: 146671642
Unique ICU stays with vital-sign records: 192831

Non-missing vital-sign measurements:
sao2           132908266
heartrate      145979794
respiration    128501032
dtype: int64


#### Observation :
The raw vital-sign table contains `146,671,642` records across `192,831` ICU stays.

Heart rate has the highest availability with `145,979,794` non-missing measurements, followed by SpO₂ with `132,908,266` and respiratory rate with `128,501,032`.

The next step is to restrict these measurements to the current `8,312` patients and the first `24` hours.

## Restrict Vital Signs to Current Cohort and First $24$ Hours

In [13]:
# Record vital-sign table information before filtering
before_shape = vitals.shape
before_records = len(vitals)
before_icu_stays = vitals["patientunitstayid"].nunique()

# Current cohort after laboratory completeness filtering
candidate_ids = set(
    sepsis_complete_labs["patientunitstayid"]
)

# Non-missing vital-sign measurement counts before filtering
vital_distribution_before = (
    vitals[["sao2", "heartrate", "respiration"]]
    .notna()
    .sum()
)

# Keep vital-sign records:
# 1. belonging to the current patient cohort
# 2. recorded during the first 24 hours after ICU admission
valid_vitals_24h = vitals[
    (vitals["patientunitstayid"].isin(candidate_ids))
    & (vitals["observationoffset"] >= 0)
    & (vitals["observationoffset"] <= 1440)
].copy()

# Record table information after filtering
after_shape = valid_vitals_24h.shape
after_records = len(valid_vitals_24h)
after_icu_stays = valid_vitals_24h["patientunitstayid"].nunique()

# Non-missing vital-sign measurement counts after filtering
vital_distribution_after = (
    valid_vitals_24h[["sao2", "heartrate", "respiration"]]
    .notna()
    .sum()
)

# Count how many patients have at least one measurement
# for each individual vital sign
patients_with_each_vital = pd.Series({
    "sao2": valid_vitals_24h.loc[
        valid_vitals_24h["sao2"].notna(),
        "patientunitstayid"
    ].nunique(),

    "heartrate": valid_vitals_24h.loc[
        valid_vitals_24h["heartrate"].notna(),
        "patientunitstayid"
    ].nunique(),

    "respiration": valid_vitals_24h.loc[
        valid_vitals_24h["respiration"].notna(),
        "patientunitstayid"
    ].nunique()
})

# ---------------------------------------------------------
# Display vital-sign extraction information
# ---------------------------------------------------------

print("Vital-sign dataset shape before filtering:", before_shape)

print("\nNon-missing vital-sign measurements before filtering:")
print(vital_distribution_before)

print("\nBefore vital-sign filtering")
print("Vital-sign records:", before_records)
print("ICU stays with vital-sign records:", before_icu_stays)

print("\nAfter restricting to current cohort and first 24 hours")
print("Vital-sign records:", after_records)
print("ICU stays with vital-sign records:", after_icu_stays)

print("\nNon-missing vital-sign measurements after filtering:")
print(vital_distribution_after)

print("\nPatients with at least one measurement of each vital sign:")
print(patients_with_each_vital)

print("\nVital-sign dataset shape after filtering:", after_shape)

Vital-sign dataset shape before filtering: (146671642, 5)

Non-missing vital-sign measurements before filtering:
sao2           132908266
heartrate      145979794
respiration    128501032
dtype: int64

Before vital-sign filtering
Vital-sign records: 146671642
ICU stays with vital-sign records: 192831

After restricting to current cohort and first 24 hours
Vital-sign records: 2169495
ICU stays with vital-sign records: 8255

Non-missing vital-sign measurements after filtering:
sao2           2003964
heartrate      2164790
respiration    2013145
dtype: int64

Patients with at least one measurement of each vital sign:
sao2           8194
heartrate      8254
respiration    7908
dtype: int64

Vital-sign dataset shape after filtering: (2169495, 5)


#### Observation :

Before vital-sign filtering, the raw table contained `146,671,642` records across `192,831` ICU stays.

After restricting to the current `8,312` patients and the first `24` hours, `2,169,495` vital-sign records remained across `8,255` ICU stays.

Heart rate was available for `8,254` patients, SpO₂ for `8,194`, and respiratory rate for `7,908`.

Respiratory rate has the lowest patient coverage and is therefore likely to remove the most patients in the next completeness step.

#### Keep Patient with All Three Vital Signs

In [14]:
# Record cohort information before vital-sign completeness filtering
before_shape = sepsis_complete_labs.shape
before_stays = len(sepsis_complete_labs)
before_patients = sepsis_complete_labs["uniquepid"].nunique()

# Create patient-level indicators showing whether each vital sign
# has at least one non-missing measurement during the first 24 hours
vital_availability = (
    valid_vitals_24h
    .groupby("patientunitstayid")[["sao2", "heartrate", "respiration"]]
    .count()
)

# Include every patient in the current cohort, even if no vital record exists
vital_availability = vital_availability.reindex(
    sepsis_complete_labs["patientunitstayid"],
    fill_value=0
)

# Number of patients with each vital sign before filtering
vital_patient_counts_before = (
    (vital_availability > 0)
    .sum()
)

# Keep patients who have at least one measurement
# of SpO2, heart rate, and respiratory rate
has_all_vitals = (vital_availability > 0).all(axis=1)

patients_with_all_vitals = vital_availability.index[has_all_vitals]

sepsis_complete_vitals = (
    sepsis_complete_labs[
        sepsis_complete_labs["patientunitstayid"].isin(
            patients_with_all_vitals
        )
    ]
    .copy()
)

# Record cohort information after filtering
after_shape = sepsis_complete_vitals.shape
after_stays = len(sepsis_complete_vitals)
after_patients = sepsis_complete_vitals["uniquepid"].nunique()

# Calculate removals
removed_stays = before_stays - after_stays
removed_patients = before_patients - after_patients

# Every retained patient now has all three vital signs
vital_patient_counts_after = pd.Series(
    after_patients,
    index=["sao2", "heartrate", "respiration"],
    name="patients"
)


# ---------------------------------------------------------
# Display cohort tracking information
# ---------------------------------------------------------

print("Dataset shape before vital-sign completeness filtering:", before_shape)

print("\nPatients with each vital sign before filtering:")
print(vital_patient_counts_before)

print("\nBefore vital-sign completeness filtering")
print("ICU stays:", before_stays)
print("Unique patients:", before_patients)

print("\nRemoved because one or more vital signs were missing")
print("ICU stays:", removed_stays)
print("Unique patients:", removed_patients)

print("\nAfter vital-sign completeness filtering")
print("ICU stays:", after_stays)
print("Unique patients:", after_patients)

print("\nPatients with each vital sign after filtering:")
print(vital_patient_counts_after)

print("\nDataset shape after vital-sign completeness filtering:", after_shape)

Dataset shape before vital-sign completeness filtering: (8312, 11)

Patients with each vital sign before filtering:
sao2           8194
heartrate      8254
respiration    7908
dtype: int64

Before vital-sign completeness filtering
ICU stays: 8312
Unique patients: 8312

Removed because one or more vital signs were missing
ICU stays: 453
Unique patients: 453

After vital-sign completeness filtering
ICU stays: 7859
Unique patients: 7859

Patients with each vital sign after filtering:
sao2           7859
heartrate      7859
respiration    7859
Name: patients, dtype: int64

Dataset shape after vital-sign completeness filtering: (7859, 11)


#### Observation :
Before vital-sign completeness filtering, the dataset contained `8,312` ICU stays from `8,312` unique patients.

A total of `453` patients were removed because at least one of SpO₂, heart rate, or respiratory rate was unavailable during the first `24` hours.

After filtering, `7,859` ICU stays from `7,859` unique patients remained.

All retained patients now have measurements for all three required vital signs.

## Create Demographic and Admission Feature Table

In [15]:
# Select demographic and admission variables that will later
# become part of the clustering feature matrix
demographic_features = sepsis_complete_vitals[
    [
        "patientunitstayid",
        "age",
        "gender",
        "ethnicity",
        "unittype",
        "admissionheight",
        "admissionweight"
    ]
].copy()

# ---------------------------------------------------------
# Check dataset size and remaining missing values
# ---------------------------------------------------------

print("Demographic feature dataset shape:", demographic_features.shape)
print("Unique patients:", demographic_features["patientunitstayid"].nunique())

print("\nMissing values:")
print(
    demographic_features
    .isna()
    .sum()
)

# ---------------------------------------------------------
# Categorical feature distributions
# ---------------------------------------------------------

print("\nGender distribution:")
print(
    demographic_features["gender"]
    .value_counts(dropna=False)
)

print("\nEthnicity distribution:")
print(
    demographic_features["ethnicity"]
    .value_counts(dropna=False)
)

print("\nICU type distribution:")
print(
    demographic_features["unittype"]
    .value_counts(dropna=False)
)

# ---------------------------------------------------------
# Numeric feature summary
# ---------------------------------------------------------

print("\nNumeric demographic/admission summary:")
print(
    demographic_features[
        ["age", "admissionheight", "admissionweight"]
    ]
    .describe()
)

Demographic feature dataset shape: (7859, 7)
Unique patients: 7859

Missing values:
patientunitstayid      0
age                    0
gender                 0
ethnicity              0
unittype               0
admissionheight       70
admissionweight      193
dtype: int64

Gender distribution:
gender
Male      3995
Female    3864
Name: count, dtype: int64

Ethnicity distribution:
ethnicity
Caucasian           6268
African American     625
Other/Unknown        352
Hispanic             340
Asian                167
Native American      107
Name: count, dtype: int64

ICU type distribution:
unittype
Med-Surg ICU    5364
MICU             872
Cardiac ICU      579
CCU-CTICU        572
SICU             230
Neuro ICU        176
CTICU             66
Name: count, dtype: int64

Numeric demographic/admission summary:
               age  admissionheight  admissionweight
count  7859.000000      7789.000000      7666.000000
mean     65.293422       168.306247        83.009947
std      16.126525        1

#### Observation :

The demographic feature table contains `7,859` unique patients.

Age, gender, ethnicity, and ICU type have no missing values, while `70` patients are missing admission height and `193` are missing admission weight.

The numeric summary also shows implausible values such as height `0 cm`, height `608 cm`, and weight `909.9 kg`, so these variables should be inspected before final preprocessing.

## Create Laboratory Summary Features

In [16]:
# Keep first 24-hour laboratory records only for the final vital-complete cohort
final_patient_ids = set(
    sepsis_complete_vitals["patientunitstayid"]
)

lab_final = valid_lab_24h[
    valid_lab_24h["patientunitstayid"].isin(final_patient_ids)
].copy()

# Calculate four summary statistics for each laboratory variable:
# minimum, maximum, mean, and standard deviation
lab_summary = (
    lab_final
    .groupby(["patientunitstayid", "labname"])["labresult"]
    .agg(["min", "max", "mean", "std"])
    .unstack("labname")
)

# Flatten the multi-level column names
lab_summary.columns = [
    f"{lab}_{stat}"
    for stat, lab in lab_summary.columns
]

# Restore patientunitstayid as a normal column
lab_summary = lab_summary.reset_index()

# Display the resulting laboratory feature table
print("Laboratory summary dataset shape:", lab_summary.shape)
print("Unique patients:", lab_summary["patientunitstayid"].nunique())

print("\nNumber of laboratory summary features:", lab_summary.shape[1] - 1)

print("\nMissing values in laboratory summary features:")
print(
    lab_summary
    .drop(columns="patientunitstayid")
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(48)
)

Laboratory summary dataset shape: (7859, 49)
Unique patients: 7859

Number of laboratory summary features: 48

Missing values in laboratory summary features:
WBC x 1000_std           5109
platelets x 1000_std     5017
Hgb_std                  4591
anion gap_std            3990
BUN_std                  3955
bicarbonate_std          3945
chloride_std             3929
creatinine_std           3924
glucose_std              3894
lactate_std              3718
sodium_std               3639
potassium_std            3119
Hgb_min                     0
sodium_mean                 0
potassium_mean              0
platelets x 1000_mean       0
lactate_mean                0
glucose_mean                0
creatinine_mean             0
chloride_mean               0
bicarbonate_mean            0
anion gap_mean              0
WBC x 1000_mean             0
Hgb_mean                    0
BUN_min                     0
sodium_max                  0
sodium_min                  0
WBC x 1000_min              0
an

#### Observation :
The laboratory summary table contains `7,859` patients and `48` features, created from `12` laboratory variables using minimum, maximum, mean, and standard deviation.

All minimum, maximum, and mean features are complete. 

Missing values appear only in the standard deviation features, which likely occurs when a patient has only one measurement for that laboratory test.

## Verify Why Laboratory Standard Deviations Are Missing

In [17]:
# Count the number of measurements available for each patient and laboratory test
lab_measurement_counts = (
    lab_final
    .groupby(["patientunitstayid", "labname"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=selected_labs, fill_value=0)
)

# Check whether every missing standard deviation corresponds
# to a patient having exactly one measurement for that lab
verification_results = {}

for lab_name in selected_labs:
    std_column = f"{lab_name}_std"

    # Patients whose standard deviation is missing
    missing_std_patients = lab_summary.loc[
        lab_summary[std_column].isna(),
        "patientunitstayid"
    ]

    # Count how many of those patients had exactly one measurement
    one_measurement_count = (
        lab_measurement_counts
        .loc[missing_std_patients, lab_name]
        .eq(1)
        .sum()
    )

    verification_results[lab_name] = {
        "Missing SD": len(missing_std_patients),
        "Exactly one measurement": one_measurement_count
    }

# Convert results to a readable table
std_verification = pd.DataFrame(verification_results).T

print("Verification of missing laboratory standard deviations:")
print(std_verification)

print(
    "\nAll missing SD values explained by one measurement:",
    (
        std_verification["Missing SD"]
        == std_verification["Exactly one measurement"]
    ).all()
)

Verification of missing laboratory standard deviations:
                  Missing SD  Exactly one measurement
bicarbonate             3945                     3945
lactate                 3718                     3718
potassium               3119                     3119
platelets x 1000        5017                     5017
anion gap               3990                     3990
chloride                3929                     3929
BUN                     3955                     3955
creatinine              3924                     3924
sodium                  3639                     3639
glucose                 3894                     3894
WBC x 1000              5109                     5109
Hgb                     4591                     4591

All missing SD values explained by one measurement: True


#### Observation :

All missing laboratory standard deviations are fully explained by patients having only one measurement for that lab.

Therefore, these are not missing clinical values; the standard deviation is simply undefined when only one observation exists.

It is appropriate to set these SD values to `0` to represent no observed within-patient variation.

## Replace Undefined Laboratory Standard Deviations With Zero

In [18]:
# Identify all laboratory standard deviation columns
std_columns = [
    column
    for column in lab_summary.columns
    if column.endswith("_std")
]

# Count missing SD values before replacement
missing_std_before = (
    lab_summary[std_columns]
    .isna()
    .sum()
    .sum()
)

# Replace undefined SD values with 0
# This is appropriate because these cases have exactly one lab measurement,
# so no within-patient variation was observed
lab_summary[std_columns] = (
    lab_summary[std_columns]
    .fillna(0)
)

# Count missing SD values after replacement
missing_std_after = (
    lab_summary[std_columns]
    .isna()
    .sum()
    .sum()
)

# Check whether any missing values remain in the full laboratory feature table
remaining_missing = (
    lab_summary
    .drop(columns="patientunitstayid")
    .isna()
    .sum()
    .sum()
)

print("Total missing SD values before replacement:", missing_std_before)
print("Total missing SD values after replacement:", missing_std_after)
print("Remaining missing values in laboratory features:", remaining_missing)
print("Laboratory summary dataset shape:", lab_summary.shape)

Total missing SD values before replacement: 48830
Total missing SD values after replacement: 0
Remaining missing values in laboratory features: 0
Laboratory summary dataset shape: (7859, 49)


#### Observation :
Before replacement, there were `48,830` undefined laboratory SD values, all caused by patients having only one measurement for the corresponding lab.

After replacing these SD values with `0`, no missing values remained in the 48 laboratory summary features.

The laboratory feature table remains `7,859 × 49`, including `patientunitstayid`.

## Create Total Laboratory Measurement Count Feature

In [19]:
# Count the total number of selected laboratory measurements
# recorded for each patient during the first 24 hours
lab_measurement_count = (
    lab_final
    .groupby("patientunitstayid")
    .size()
    .rename("lab_measurement_count")
    .reset_index()
)

# Check the resulting feature table
print("Lab measurement count dataset shape:", lab_measurement_count.shape)
print("Unique patients:", lab_measurement_count["patientunitstayid"].nunique())

print("\nLab measurement count summary:")
print(
    lab_measurement_count["lab_measurement_count"]
    .describe()
)

print(
    "\nMost common lab measurement count:",
    lab_measurement_count["lab_measurement_count"].mode().iloc[0]
)

Lab measurement count dataset shape: (7859, 2)
Unique patients: 7859

Lab measurement count summary:
count    7859.000000
mean       21.894643
std        11.268325
min        12.000000
25%        13.000000
50%        20.000000
75%        26.000000
max       146.000000
Name: lab_measurement_count, dtype: float64

Most common lab measurement count: 12


#### Observation :
The lab measurement count feature contains all `7,859` patients.

Patients had between `12` and `146` selected laboratory measurements during the first 24 hours, with a median of `20`.

The most common count was `12`, meaning many patients had only one measurement for each of the 12 selected laboratory variables.

## Restrict Vital-Sign Records to the Final $7,859$ Patient Cohort

In [20]:
# Keep only vital-sign records belonging to the final cohort
final_patient_ids = set(
    sepsis_complete_vitals["patientunitstayid"]
)

vitals_final = valid_vitals_24h[
    valid_vitals_24h["patientunitstayid"].isin(final_patient_ids)
].copy()

# Check the final vital-sign table
print("Final vital-sign dataset shape:", vitals_final.shape)
print("Unique ICU stays:", vitals_final["patientunitstayid"].nunique())

print("\nNon-missing vital-sign measurements:")
print(
    vitals_final[
        ["sao2", "heartrate", "respiration"]
    ]
    .notna()
    .sum()
)

Final vital-sign dataset shape: (2066966, 5)
Unique ICU stays: 7859

Non-missing vital-sign measurements:
sao2           1924100
heartrate      2062753
respiration    2001194
dtype: int64


#### Observation :
The final vital-sign table contains `2,066,966` first-24-hour records from all `7,859` retained patients.

Heart rate has `2,062,753` valid measurements, respiratory rate has `2,001,194`, and SpO₂ has `1,924,100`.

All `7,859` patients are represented, so the data are ready for functional trajectory construction.

## Convert SpO2 Measurements into Irregular Functional Data

In [21]:
# Fix one patient order so all later FPCA features remain aligned
patient_order = (
    sepsis_complete_vitals["patientunitstayid"]
    .tolist()
)

# Keep only observed SpO2 measurements
sao2_data = vitals_final[
    vitals_final["sao2"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "sao2"
    ]
].copy()

# Sort each patients SpO2 measurements in chronological order
sao2_data = sao2_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

# Group measurements by patient
sao2_groups = sao2_data.groupby(
    "patientunitstayid"
)

# Dictionaries required by FDApy for irregular functional data
sao2_argvals_dict = {}
sao2_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = sao2_groups.get_group(patient_id)

    # Measurement times during the first 24 hours
    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    # Corresponding SpO2 values
    values = patient_data[
        "sao2"
    ].to_numpy(dtype=float)

    # Store each patients irregular observation times
    sao2_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    # Store each patients observed SpO2 values
    sao2_values_dict[i] = values


# Combine patient-specific times and measurements
# into one irregular functional dataset
sao2_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        sao2_argvals_dict
    ),
    values=IrregularValues(
        sao2_values_dict
    )
)

# Verify the functional dataset
print("Patients in SpO2 functional data:", sao2_fdata.n_obs)
print("Functional dimensions:", sao2_fdata.n_dimension)

print("\nFirst patient measurements:", len(sao2_values_dict[0]))

print(
    "First patient time range:",
    sao2_argvals_dict[0]["input_dim_0"].min(),
    "to",
    sao2_argvals_dict[0]["input_dim_0"].max()
)

Patients in SpO2 functional data: 7859
Functional dimensions: 1

First patient measurements: 288
First patient time range: 5.0 to 1440.0


#### Observation :

The SpO₂ functional dataset contains all `7,859` patients and is correctly represented as one-dimensional irregular functional data.

For the first patient, `288` SpO₂ measurements were available from minute `5` to minute `1440`, confirming dense first-24-hour coverage suitable for functional PCA.

## Fit UFPCA to SpO2 Trajectories

In [22]:
# Fit functional PCA using covariance decomposition
# n_components=0.90 keeps enough components to explain at least 90% of SpO2 variation
sao2_fpca = UFPCA(
    method="covariance",
    n_components=0.90
)

# Estimate the mean function, covariance structure,
# eigenvalues, and eigenfunctions from irregular SpO2 trajectories
# PS = Penalized Spline smoothing
sao2_fpca.fit(
    sao2_fdata,
    method_smoothing="PS"
)

# Calculate patient-specific functional principal component scores
# PACE is designed for irregular longitudinal measurements
sao2_scores = sao2_fpca.transform(
    sao2_fdata,
    method="PACE",
    method_smoothing="PS"
)

# Display FPCA results
print("SpO2 FPCA score matrix shape:", sao2_scores.shape)
print("Number of patients:", sao2_scores.shape[0])
print("Number of retained SpO2 FPCs:", sao2_scores.shape[1])

# FDApy stores the variance associated with retained components as eigenvalues
print("\nRetained eigenvalues:")
print(sao2_fpca.eigenvalues)

print("\nVariance retention target: 90%")

SpO2 FPCA score matrix shape: (7859, 8)
Number of patients: 7859
Number of retained SpO2 FPCs: 8

Retained eigenvalues:
[8126.40360108 1822.50025107 1163.35215848  826.13851741  636.2173004
  436.34756691  356.14660506  276.40185264]

Variance retention target: 90%


#### Observation :

The SpO₂ FPCA successfully processed all `7,859` patients.

A total of `8` functional principal components were retained to capture at least `90%` of the variation in the 24 hour SpO₂ trajectories.

These `8` FPC scores will now serve as compact numerical features representing each patients SpO₂ pattern over time.

## Create SpO2 FPC Feature Table

In [23]:
# Create readable feature names for the retained SpO2 components
sao2_feature_names = [
    f"sao2_fpc_{i+1}"
    for i in range(sao2_scores.shape[1])
]

# Convert the FPCA score matrix into a DataFrame
# Each row represents one patient
# Each column represents one retained SpO2 functional principal component
sao2_features = pd.DataFrame(
    sao2_scores,
    columns=sao2_feature_names
)

# Add patientunitstayid so the FPCA features can later be merged
# with demographic, laboratory, and other vital-sign features
sao2_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Display the resulting feature table information
print("SpO2 feature dataset shape:", sao2_features.shape)
print("Unique patients:", sao2_features["patientunitstayid"].nunique())
print("Number of SpO2 FPC features:", len(sao2_feature_names))

print("\nMissing values in SpO2 FPC features:")
print(
    sao2_features[sao2_feature_names]
    .isna()
    .sum()
)

print("\nFirst five rows:")
sao2_features.head()

SpO2 feature dataset shape: (7859, 9)
Unique patients: 7859
Number of SpO2 FPC features: 8

Missing values in SpO2 FPC features:
sao2_fpc_1    0
sao2_fpc_2    0
sao2_fpc_3    0
sao2_fpc_4    0
sao2_fpc_5    0
sao2_fpc_6    0
sao2_fpc_7    0
sao2_fpc_8    0
dtype: int64

First five rows:


,patientunitstayid,sao2_fpc_1,sao2_fpc_2,sao2_fpc_3,sao2_fpc_4,sao2_fpc_5,sao2_fpc_6,sao2_fpc_7,sao2_fpc_8
0,151900,-48.001631,78.186428,33.683153,-13.502005,-20.045747,10.059184,-3.351875,9.248180
1,165269,-6.719933,-19.476389,12.883028,-24.860240,-7.792970,28.752334,-4.069120,3.096719
2,179269,129.908675,-49.710517,-11.118698,-3.894689,-1.108633,33.321812,-7.433772,-1.858052
3,172764,-63.083806,-14.484540,-16.114482,-2.313732,-10.952623,-4.527089,-6.591582,10.298815
4,215156,-44.069719,-22.205449,41.479366,23.521806,0.100980,17.765286,-0.111048,-4.723925


#### Observation :
The SpO₂ feature table contains all `7,859` patients and `8` FPC features.

No missing values are present, so every retained patient has a complete SpO₂ functional representation ready for the final clustering matrix.

## Convert Heart Rate Measurements Into Irregular Functional Data

In [24]:
# Keep only observed heart rate measurements
heartrate_data = vitals_final[
    vitals_final["heartrate"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "heartrate"
    ]
].copy()

# Sort each patients heart rate measurements in chronological order
heartrate_data = heartrate_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

# Group measurements by patient
heartrate_groups = heartrate_data.groupby(
    "patientunitstayid"
)

# Dictionaries required by FDApy for irregular functional data
heartrate_argvals_dict = {}
heartrate_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = heartrate_groups.get_group(patient_id)

    # Measurement times during the first 24 hours
    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    # Corresponding heart rate values
    values = patient_data[
        "heartrate"
    ].to_numpy(dtype=float)

    # Store irregular observation times
    heartrate_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    # Store observed heart rate values
    heartrate_values_dict[i] = values


# Combine patient-specific times and measurements
# into one irregular functional dataset
heartrate_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        heartrate_argvals_dict
    ),
    values=IrregularValues(
        heartrate_values_dict
    )
)

# Verify the functional dataset
print("Patients in heart rate functional data:", heartrate_fdata.n_obs)
print("Functional dimensions:", heartrate_fdata.n_dimension)

print("\nFirst patient measurements:", len(heartrate_values_dict[0]))

print(
    "First patient time range:",
    heartrate_argvals_dict[0]["input_dim_0"].min(),
    "to",
    heartrate_argvals_dict[0]["input_dim_0"].max()
)

Patients in heart rate functional data: 7859
Functional dimensions: 1

First patient measurements: 288
First patient time range: 5.0 to 1440.0


#### Observation :

The heart rate functional dataset contains all `7,859` patients and is correctly represented as one-dimensional irregular functional data.

For the first patient, `288` heart rate measurements were available from minute `5 to 1440`, confirming complete first 24 hour trajectory coverage.

## Fit Heart Rate UFPCA and Calculate PACE Scores

In [25]:
# Fit functional PCA using covariance decomposition
# Keep enough components to explain at least 90% of heart rate variation
heartrate_fpca = UFPCA(
    method="covariance",
    n_components=0.90
)

# Estimate the mean function, covariance structure,
# eigenvalues, and eigenfunctions using Penalized Spline smoothing
heartrate_fpca.fit(
    heartrate_fdata,
    method_smoothing="PS"
)

# Calculate patient-specific functional principal component scores
# using PACE for irregular longitudinal measurements
heartrate_scores = heartrate_fpca.transform(
    heartrate_fdata,
    method="PACE",
    method_smoothing="PS"
)

# Display FPCA results
print("Heart rate FPCA score matrix shape:", heartrate_scores.shape)
print("Number of patients:", heartrate_scores.shape[0])
print("Number of retained heart rate FPCs:", heartrate_scores.shape[1])

print("\nRetained eigenvalues:")
print(heartrate_fpca.eigenvalues)

print("\nVariance retention target: 90%")

Heart rate FPCA score matrix shape: (7859, 2)
Number of patients: 7859
Number of retained heart rate FPCs: 2

Retained eigenvalues:
[408286.34428004  50637.4945104 ]

Variance retention target: 90%


#### Observation :

The heart rate FPCA successfully processed all `7,859` patients.

A total of `2` functional principal components were retained to capture at least `90%` of the variation in the 24 hour heart rate trajectories.

These `2` FPC scores will represent each patients heart rate pattern in the clustering dataset.

## Create Heart Rate FPC Feature Table

In [26]:
# Create readable names for the retained heart rate components
heartrate_feature_names = [
    f"heartrate_fpc_{i+1}"
    for i in range(heartrate_scores.shape[1])
]

# Convert the FPCA score matrix into a DataFrame
heartrate_features = pd.DataFrame(
    heartrate_scores,
    columns=heartrate_feature_names
)

# Add patientunitstayid so these features can be merged later
heartrate_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Display the resulting feature table information
print("Heart rate feature dataset shape:", heartrate_features.shape)
print("Unique patients:", heartrate_features["patientunitstayid"].nunique())
print("Number of heart rate FPC features:", len(heartrate_feature_names))

print("\nMissing values in heart rate FPC features:")
print(
    heartrate_features[heartrate_feature_names]
    .isna()
    .sum()
)

print("\nFirst five rows:")
heartrate_features.head()

Heart rate feature dataset shape: (7859, 3)
Unique patients: 7859
Number of heart rate FPC features: 2

Missing values in heart rate FPC features:
heartrate_fpc_1    0
heartrate_fpc_2    0
dtype: int64

First five rows:


,patientunitstayid,heartrate_fpc_1,heartrate_fpc_2
0,151900,321.262146,242.651827
1,165269,5.596177,103.202770
2,179269,-112.794604,-167.366156
3,172764,-1015.959049,-174.007248
4,215156,-753.324677,-420.767067


#### Observation :
The heart rate feature table contains all `7,859` patients and `2` FPC features.

No missing values are present, so every retained patient has a complete heart rate functional representation ready for the final clustering matrix.

## Convert Respiratory Rate Measurements Into Irregular Functional Data

In [27]:
# Keep only observed respiratory rate measurements
respiration_data = vitals_final[
    vitals_final["respiration"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "respiration"
    ]
].copy()

# Sort each patients respiratory rate measurements in chronological order
respiration_data = respiration_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

# Group respiratory rate measurements by patient
respiration_groups = respiration_data.groupby(
    "patientunitstayid"
)

# Dictionaries required by FDApy for irregular functional data
respiration_argvals_dict = {}
respiration_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = respiration_groups.get_group(patient_id)

    # Measurement times during the first 24 hours
    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    # Corresponding respiratory rate values
    values = patient_data[
        "respiration"
    ].to_numpy(dtype=float)

    # Store irregular observation times
    respiration_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    # Store observed respiratory rate values
    respiration_values_dict[i] = values


# Combine patient-specific times and measurements
# into one irregular functional dataset
respiration_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        respiration_argvals_dict
    ),
    values=IrregularValues(
        respiration_values_dict
    )
)

# Verify the functional dataset
print("Patients in respiratory rate functional data:", respiration_fdata.n_obs)
print("Functional dimensions:", respiration_fdata.n_dimension)

print("\nFirst patient measurements:", len(respiration_values_dict[0]))

print(
    "First patient time range:",
    respiration_argvals_dict[0]["input_dim_0"].min(),
    "to",
    respiration_argvals_dict[0]["input_dim_0"].max()
)

Patients in respiratory rate functional data: 7859
Functional dimensions: 1

First patient measurements: 75
First patient time range: 1070.0 to 1440.0


#### Observation :

The respiratory rate functional dataset contains all `7,859` patients.

For the first patient, only `75` respiratory rate measurements were available, from minute `1070 to 1440`, showing that some trajectories are sparse and cover only part of the first 24 hours.

This is exactly why irregular functional data and PACE are useful here.

## Fit Respiratory Rate UFPCA and Calculate PACE Scores

In [28]:
# Fit functional PCA using covariance decomposition
# Keep enough components to explain at least 90% of respiratory rate variation
respiration_fpca = UFPCA(
    method="covariance",
    n_components=0.90
)

# Estimate the mean function, covariance structure,
# eigenvalues, and eigenfunctions using Penalized Spline smoothing
respiration_fpca.fit(
    respiration_fdata,
    method_smoothing="PS"
)

# Calculate patient-specific functional principal component scores
# using PACE for irregular longitudinal measurements
respiration_scores = respiration_fpca.transform(
    respiration_fdata,
    method="PACE",
    method_smoothing="PS"
)

# Display FPCA results
print("Respiratory rate FPCA score matrix shape:", respiration_scores.shape)
print("Number of patients:", respiration_scores.shape[0])
print("Number of retained respiratory rate FPCs:", respiration_scores.shape[1])

print("\nRetained eigenvalues:")
print(respiration_fpca.eigenvalues)

print("\nVariance retention target: 90%")

Respiratory rate FPCA score matrix shape: (7859, 4)
Number of patients: 7859
Number of retained respiratory rate FPCs: 4

Retained eigenvalues:
[38224.41266092  5564.12557104  3231.88502961  1547.48852609]

Variance retention target: 90%


#### Observation

The respiratory rate FPCA successfully processed all `7,859` patients.

A total of `4` functional principal components were retained to capture at least 90% of the variation in the 24 hour respiratory rate trajectories.

These `4` FPC scores will represent each patients respiratory pattern in the clustering dataset.

## Create Respiratory Rate FPC Feature Table

In [29]:
# Create readable names for the retained respiratory rate components
respiration_feature_names = [
    f"respiration_fpc_{i+1}"
    for i in range(respiration_scores.shape[1])
]

# Convert the FPCA score matrix into a DataFrame
respiration_features = pd.DataFrame(
    respiration_scores,
    columns=respiration_feature_names
)

# Add patientunitstayid so these features can be merged later
respiration_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Display the resulting feature table information
print("Respiratory rate feature dataset shape:", respiration_features.shape)
print("Unique patients:", respiration_features["patientunitstayid"].nunique())
print("Number of respiratory rate FPC features:", len(respiration_feature_names))

print("\nMissing values in respiratory rate FPC features:")
print(
    respiration_features[respiration_feature_names]
    .isna()
    .sum()
)

print("\nFirst five rows:")

respiration_features.head()

Respiratory rate feature dataset shape: (7859, 5)
Unique patients: 7859
Number of respiratory rate FPC features: 4

Missing values in respiratory rate FPC features:
respiration_fpc_1    0
respiration_fpc_2    0
respiration_fpc_3    0
respiration_fpc_4    0
dtype: int64

First five rows:


,patientunitstayid,respiration_fpc_1,respiration_fpc_2,respiration_fpc_3,respiration_fpc_4
0,151900,-293.479735,-30.989069,9.878976,-19.137185
1,165269,75.662554,-3.345398,-93.678218,10.426874
2,179269,80.651845,13.177399,19.741218,-81.337420
3,172764,206.370922,190.054600,-4.986728,-7.197024
4,215156,292.126388,79.045576,30.309833,47.254615


#### Observation :

The respiratory rate feature table contains all `7,859` patients and `4` FPC features.

No missing values are present, so every retained patient has a complete respiratory rate functional representation ready for the final clustering matrix.

## Combining All Vital-Sign FPC Features

In [30]:
# Merge SpO2, heart rate, and respiratory rate FPC features
# using patientunitstayid as the common patient identifier
vital_fpc_features = (
    sao2_features
    .merge(
        heartrate_features,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        respiration_features,
        on="patientunitstayid",
        how="inner"
    )
)

# Count the total number of functional features
total_fpc_features = (
    len(sao2_feature_names)
    + len(heartrate_feature_names)
    + len(respiration_feature_names)
)

# Check the combined vital-sign feature table
print("Combined vital FPC dataset shape:", vital_fpc_features.shape)
print("Unique patients:", vital_fpc_features["patientunitstayid"].nunique())

print("\nSpO2 FPC features:", len(sao2_feature_names))
print("Heart rate FPC features:", len(heartrate_feature_names))
print("Respiratory rate FPC features:", len(respiration_feature_names))
print("Total vital-sign FPC features:", total_fpc_features)

print("\nTotal missing values:")
print(
    vital_fpc_features
    .drop(columns="patientunitstayid")
    .isna()
    .sum()
    .sum()
)

print("\nFirst five rows:")
vital_fpc_features.head()

Combined vital FPC dataset shape: (7859, 15)
Unique patients: 7859

SpO2 FPC features: 8
Heart rate FPC features: 2
Respiratory rate FPC features: 4
Total vital-sign FPC features: 14

Total missing values:
0

First five rows:


,patientunitstayid,sao2_fpc_1,sao2_fpc_2,sao2_fpc_3,sao2_fpc_4,sao2_fpc_5,sao2_fpc_6,sao2_fpc_7,sao2_fpc_8,heartrate_fpc_1,heartrate_fpc_2,respiration_fpc_1,respiration_fpc_2,respiration_fpc_3,respiration_fpc_4
0,151900,-48.001631,78.186428,33.683153,-13.502005,-20.045747,10.059184,-3.351875,9.248180,321.262146,242.651827,-293.479735,-30.989069,9.878976,-19.137185
1,165269,-6.719933,-19.476389,12.883028,-24.860240,-7.792970,28.752334,-4.069120,3.096719,5.596177,103.202770,75.662554,-3.345398,-93.678218,10.426874
2,179269,129.908675,-49.710517,-11.118698,-3.894689,-1.108633,33.321812,-7.433772,-1.858052,-112.794604,-167.366156,80.651845,13.177399,19.741218,-81.337420
3,172764,-63.083806,-14.484540,-16.114482,-2.313732,-10.952623,-4.527089,-6.591582,10.298815,-1015.959049,-174.007248,206.370922,190.054600,-4.986728,-7.197024
4,215156,-44.069719,-22.205449,41.479366,23.521806,0.100980,17.765286,-0.111048,-4.723925,-753.324677,-420.767067,292.126388,79.045576,30.309833,47.254615


#### Observation :

The combined vital-sign feature table contains all `7,859` patients and `14` functional features: `8 SpO₂ FPCs`, `2 heart rate FPCs`, and `4 respiratory rate FPCs`.

There are no missing values, confirming that all three vital-sign representations can be safely merged into the patient-level feature dataset.

## Combine Demographic, Laboratory, Lab Count and Vital-Sign Features

In [31]:
# Merge all constructed feature groups into one patient-level dataset
# Each row should represent one ICU patient

feature_data = (
    demographic_features
    .merge(
        lab_summary,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        lab_measurement_count,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        vital_fpc_features,
        on="patientunitstayid",
        how="inner"
    )
)

# Count patients and features after merging
n_patients = feature_data["patientunitstayid"].nunique()
n_features = feature_data.shape[1] - 1

# Check missing values after combining all feature groups
missing_summary = (
    feature_data
    .isna()
    .sum()
)

missing_summary = missing_summary[
    missing_summary > 0
].sort_values(ascending=False)

# ---------------------------------------------------------
# Display combined feature dataset information
# ---------------------------------------------------------

print("Combined feature dataset shape:", feature_data.shape)
print("Unique patients:", n_patients)
print("Number of candidate features:", n_features)

print("\nPatients before merging:", len(demographic_features))
print("Patients after merging:", len(feature_data))
print("Patients lost during merging:", len(demographic_features) - len(feature_data))

print("\nFeatures with remaining missing values:")
print(missing_summary)

print(
    "\nTotal missing values:",
    feature_data.isna().sum().sum()
)

Combined feature dataset shape: (7859, 70)
Unique patients: 7859
Number of candidate features: 69

Patients before merging: 7859
Patients after merging: 7859
Patients lost during merging: 0

Features with remaining missing values:
admissionweight    193
admissionheight     70
dtype: int64

Total missing values: 263


#### Observation :

The combined feature dataset contains `7,859` unique patients and `69` candidate features.

No patients were lost while merging demographic, laboratory, lab-count, and vital-sign FPC features.

Only `263` missing values remain: `193` in admission weight and `70` in admission height. All other features are complete.

## Inspect Remaining Height and Weight Missingness

In [32]:
# Record the current dataset size before handling height and weight missingness
before_shape = feature_data.shape
before_patients = feature_data["patientunitstayid"].nunique()

# Identify missing height and weight values
missing_height = feature_data["admissionheight"].isna()
missing_weight = feature_data["admissionweight"].isna()

# Count the different missingness patterns
height_only = (missing_height & ~missing_weight).sum()
weight_only = (~missing_height & missing_weight).sum()
both_missing = (missing_height & missing_weight).sum()
any_missing = (missing_height | missing_weight).sum()

# Display remaining missingness information
print("Dataset shape before height/weight treatment:", before_shape)
print("Unique patients:", before_patients)

print("\nMissing admission height:", missing_height.sum())
print("Missing admission weight:", missing_weight.sum())

print("\nMissingness pattern:")
print("Height only missing:", height_only)
print("Weight only missing:", weight_only)
print("Both height and weight missing:", both_missing)
print("Patients missing at least one:", any_missing)

print("\nAdmission height summary:")
print(feature_data["admissionheight"].describe())

print("\nAdmission weight summary:")
print(feature_data["admissionweight"].describe())

Dataset shape before height/weight treatment: (7859, 70)
Unique patients: 7859

Missing admission height: 70
Missing admission weight: 193

Missingness pattern:
Height only missing: 41
Weight only missing: 164
Both height and weight missing: 29
Patients missing at least one: 234

Admission height summary:
count    7789.000000
mean      168.306247
std        14.205999
min         0.000000
25%       160.000000
50%       168.000000
75%       177.800000
max       608.000000
Name: admissionheight, dtype: float64

Admission weight summary:
count    7666.000000
mean       83.009947
std        30.168293
min        18.100000
25%        63.500000
50%        77.200000
75%        95.700000
max       909.900000
Name: admissionweight, dtype: float64


#### Observation :

Before height and weight treatment, the dataset contains `7,859` unique patients and `69` candidate features.

There are `70` missing height values and `193` missing weight values, affecting `234` patients in total because `29` patients are missing both.

The minimum and maximum values also suggest possible data-quality issues, especially height `0 cm`, height `608 cm`, and weight `909.9 kg`. These should be inspected before deciding whether to remove or impute anything.

## Inspect Extreme Height and Weight Values

In [33]:
# Inspect extreme recorded values before making any preprocessing decision
# No patients or values are changed in this cell

print("10 smallest admission heights:")
print(
    feature_data["admissionheight"]
    .dropna()
    .sort_values()
    .head(10)
    .to_list()
)

print("\n10 largest admission heights:")
print(
    feature_data["admissionheight"]
    .dropna()
    .sort_values(ascending=False)
    .head(10)
    .to_list()
)

print("\n10 smallest admission weights:")
print(
    feature_data["admissionweight"]
    .dropna()
    .sort_values()
    .head(10)
    .to_list()
)

print("\n10 largest admission weights:")
print(
    feature_data["admissionweight"]
    .dropna()
    .sort_values(ascending=False)
    .head(10)
    .to_list()
)

10 smallest admission heights:
[0.0, 0.0, 1.65, 1.67, 1.7, 1.72, 1.8, 1.8, 12.7, 15.2]

10 largest admission heights:
[608.0, 248.9, 213.0, 203.2, 202.0, 200.7, 200.7, 200.7, 200.6, 198.1]

10 smallest admission weights:
[18.1, 19.8, 24.7, 25.4, 27.21, 28.2, 28.6, 29.4, 29.5, 29.5]

10 largest admission weights:
[909.9, 606.0, 303.0, 285.8, 282.0, 264.3, 262.4, 250.0, 249.9, 245.3]


#### Observation :

Height and weight contain a small number of clearly unusual values, such as `0 cm`, `608 cm`, and `909.9 kg`.

These should not be automatically removed or corrected, because we do not have enough evidence to determine the true values.

## Quantify Missing Height and Weight Before Treatment

In [34]:
# Record dataset information before handling height and weight
before_shape = feature_data.shape
before_patients = feature_data["patientunitstayid"].nunique()

# Identify patients missing height, weight, or both
missing_height = feature_data["admissionheight"].isna()
missing_weight = feature_data["admissionweight"].isna()
missing_any = missing_height | missing_weight

# Count patients affected
patients_missing_height = missing_height.sum()
patients_missing_weight = missing_weight.sum()
patients_missing_any = missing_any.sum()

# Calculate percentage of the cohort affected
height_missing_pct = patients_missing_height / before_patients * 100
weight_missing_pct = patients_missing_weight / before_patients * 100
any_missing_pct = patients_missing_any / before_patients * 100

# Display missingness before deciding on the treatment
print("Dataset shape before height/weight treatment:", before_shape)
print("Unique patients:", before_patients)

print("\nHeight missing:")
print("Patients:", patients_missing_height)
print("Percentage:", round(height_missing_pct, 2), "%")

print("\nWeight missing:")
print("Patients:", patients_missing_weight)
print("Percentage:", round(weight_missing_pct, 2), "%")

print("\nMissing height or weight:")
print("Patients:", patients_missing_any)
print("Percentage:", round(any_missing_pct, 2), "%")

print("\nPatients with complete height and weight:")
print(before_patients - patients_missing_any)

Dataset shape before height/weight treatment: (7859, 70)
Unique patients: 7859

Height missing:
Patients: 70
Percentage: 0.89 %

Weight missing:
Patients: 193
Percentage: 2.46 %

Missing height or weight:
Patients: 234
Percentage: 2.98 %

Patients with complete height and weight:
7625


#### Observation :

Before height and weight treatment, the dataset contains `7,859` unique patients.

Height is missing for `70 patients (0.89%)` and weight for `193 patients (2.46%)`. 

Overall, `234 patients (2.98%)` are missing at least one of these variables.

Because the missing proportion is small, median imputation can retain these patients instead of removing them.

## Impute Missing Height and Weight With Median Values

In [35]:
# Record dataset information before imputation
before_shape = feature_data.shape
before_patients = feature_data["patientunitstayid"].nunique()

# Calculate median values using only observed measurements
height_median = feature_data["admissionheight"].median()
weight_median = feature_data["admissionweight"].median()

# Count missing values before imputation
height_missing_before = feature_data["admissionheight"].isna().sum()
weight_missing_before = feature_data["admissionweight"].isna().sum()

# Create a copy so the original combined dataset remains unchanged
feature_data_imputed = feature_data.copy()

# Replace missing height and weight values with the corresponding median
feature_data_imputed["admissionheight"] = (
    feature_data_imputed["admissionheight"]
    .fillna(height_median)
)

feature_data_imputed["admissionweight"] = (
    feature_data_imputed["admissionweight"]
    .fillna(weight_median)
)

# Count missing values after imputation
height_missing_after = feature_data_imputed["admissionheight"].isna().sum()
weight_missing_after = feature_data_imputed["admissionweight"].isna().sum()

# ---------------------------------------------------------
# Display preprocessing information
# ---------------------------------------------------------

print("Dataset shape before imputation:", before_shape)
print("Unique patients before imputation:", before_patients)

print("\nMedian values used for imputation:")
print("Admission height median:", height_median)
print("Admission weight median:", weight_median)

print("\nMissing values before imputation:")
print("Admission height:", height_missing_before)
print("Admission weight:", weight_missing_before)

print("\nMissing values after imputation:")
print("Admission height:", height_missing_after)
print("Admission weight:", weight_missing_after)

print("\nDataset shape after imputation:", feature_data_imputed.shape)
print(
    "Unique patients after imputation:",
    feature_data_imputed["patientunitstayid"].nunique()
)

print(
    "\nTotal remaining missing values:",
    feature_data_imputed.isna().sum().sum()
)

Dataset shape before imputation: (7859, 70)
Unique patients before imputation: 7859

Median values used for imputation:
Admission height median: 168.0
Admission weight median: 77.2

Missing values before imputation:
Admission height: 70
Admission weight: 193

Missing values after imputation:
Admission height: 0
Admission weight: 0

Dataset shape after imputation: (7859, 70)
Unique patients after imputation: 7859

Total remaining missing values: 0


#### Observation :

Before imputation, the dataset contained `7,859` patients with `70` missing height values and `193` missing weight values.

Median values of `168.0 cm` for height and `77.2 kg` for weight were used to fill the missing values.

After imputation, all `7,859` patients were retained and the dataset contains no missing values.

## Encode Categorical Variables

In [36]:
# Record dataset information before categorical encoding
before_shape = feature_data_imputed.shape
before_patients = feature_data_imputed["patientunitstayid"].nunique()

# Display categorical distributions before encoding
print("Dataset shape before categorical encoding:", before_shape)

print("\nGender distribution before encoding:")
print(
    feature_data_imputed["gender"]
    .value_counts()
)

print("\nEthnicity distribution before encoding:")
print(
    feature_data_imputed["ethnicity"]
    .value_counts()
)

print("\nICU type distribution before encoding:")
print(
    feature_data_imputed["unittype"]
    .value_counts()
)

# Create a copy for encoding
feature_data_encoded = feature_data_imputed.copy()

# Convert gender into a numeric binary variable
feature_data_encoded["gender"] = (
    feature_data_encoded["gender"]
    .map({
        "Female": 0,
        "Male": 1
    })
)

# One-hot encode ethnicity and ICU type
# dtype=int keeps the dummy variables as 0 and 1
feature_data_encoded = pd.get_dummies(
    feature_data_encoded,
    columns=[
        "ethnicity",
        "unittype"
    ],
    dtype=int
)

# Record dataset information after encoding
after_shape = feature_data_encoded.shape
after_patients = feature_data_encoded["patientunitstayid"].nunique()

# Identify newly created categorical columns
encoded_columns = [
    column
    for column in feature_data_encoded.columns
    if column.startswith("ethnicity_")
    or column.startswith("unittype_")
]

print("\nAfter categorical encoding")
print("Dataset shape:", after_shape)
print("Unique patients:", after_patients)

print("\nEncoded gender distribution:")
print(
    feature_data_encoded["gender"]
    .value_counts()
    .sort_index()
)

print("\nOne-hot encoded columns:")
print(encoded_columns)

print(
    "\nTotal missing values after encoding:",
    feature_data_encoded.isna().sum().sum()
)

Dataset shape before categorical encoding: (7859, 70)

Gender distribution before encoding:
gender
Male      3995
Female    3864
Name: count, dtype: int64

Ethnicity distribution before encoding:
ethnicity
Caucasian           6268
African American     625
Other/Unknown        352
Hispanic             340
Asian                167
Native American      107
Name: count, dtype: int64

ICU type distribution before encoding:
unittype
Med-Surg ICU    5364
MICU             872
Cardiac ICU      579
CCU-CTICU        572
SICU             230
Neuro ICU        176
CTICU             66
Name: count, dtype: int64

After categorical encoding
Dataset shape: (7859, 81)
Unique patients: 7859

Encoded gender distribution:
gender
0    3864
1    3995
Name: count, dtype: int64

One-hot encoded columns:
['ethnicity_African American', 'ethnicity_Asian', 'ethnicity_Caucasian', 'ethnicity_Hispanic', 'ethnicity_Native American', 'ethnicity_Other/Unknown', 'unittype_CCU-CTICU', 'unittype_CTICU', 'unittype_Cardiac IC

#### Observation :

Before encoding, the dataset contained `7,859` patients and `70` columns.

Gender was converted to a binary numeric feature with `0 = Female` and `1 = Male`.

Ethnicity and ICU type were one-hot encoded into `13` binary columns.

After encoding, the dataset contains `7,859` patients and `81` columns, with no missing values.

## Min-Max Scale the Clustering Features

In [37]:
# Record dataset information before scaling
before_shape = feature_data_encoded.shape
before_patients = feature_data_encoded["patientunitstayid"].nunique()

# Separate the patient identifier from clustering features
patient_ids = feature_data_encoded["patientunitstayid"].copy()

clustering_features = feature_data_encoded.drop(
    columns="patientunitstayid"
)

# Store the feature names in their current order
feature_names = clustering_features.columns.tolist()

# Apply Min-Max scaling so every clustering feature is placed
# approximately on the same 0 to 1 scale
scaler = MinMaxScaler()

scaled_array = scaler.fit_transform(
    clustering_features
)

# Convert the scaled matrix back into a DataFrame
scaled_features = pd.DataFrame(
    scaled_array,
    columns=feature_names
)

# Add patientunitstayid back for tracking and later merging
scaled_features.insert(
    0,
    "patientunitstayid",
    patient_ids.to_numpy()
)

# ---------------------------------------------------------
# Display scaling information
# ---------------------------------------------------------

print("Dataset shape before scaling:", before_shape)
print("Unique patients before scaling:", before_patients)
print("Number of clustering features:", len(feature_names))

print("\nFeature value range before scaling:")
print("Minimum:", clustering_features.min().min())
print("Maximum:", clustering_features.max().max())

print("\nDataset shape after scaling:", scaled_features.shape)
print(
    "Unique patients after scaling:",
    scaled_features["patientunitstayid"].nunique()
)

print("\nScaled feature value range:")
print(
    "Minimum:",
    scaled_features.drop(columns="patientunitstayid").min().min()
)
print(
    "Maximum:",
    scaled_features.drop(columns="patientunitstayid").max().max()
)

print(
    "\nTotal missing values after scaling:",
    scaled_features.isna().sum().sum()
)

Dataset shape before scaling: (7859, 81)
Unique patients before scaling: 7859
Number of clustering features: 80

Feature value range before scaling:
Minimum: -3034.733728355761
Maximum: 3528.6048470903597

Dataset shape after scaling: (7859, 81)
Unique patients after scaling: 7859

Scaled feature value range:
Minimum: 0.0
Maximum: 1.0000000000000002

Total missing values after scaling: 0


#### Observation :

Before scaling, the dataset contained `7,859` patients and `80` clustering features with values ranging from about `-3034.73` to `3528.60`.

After Min-Max scaling, all `80` features were transformed to approximately the `0–1` range.

No patients were removed and no missing values were introduced, so the scaled clustering matrix remains `7,859 × 81`, including `patientunitstayid`.

## Final Dataset Validation

In [38]:
# Final checks before saving the clustering dataset

# Separate patient identifier from clustering features
final_feature_matrix = scaled_features.drop(
    columns="patientunitstayid"
)

print("Final dataset shape:", scaled_features.shape)
print("Unique patients:", scaled_features["patientunitstayid"].nunique())
print("Duplicate patient IDs:", scaled_features["patientunitstayid"].duplicated().sum())

print("\nNumber of clustering features:", final_feature_matrix.shape[1])

print("\nMissing values:", final_feature_matrix.isna().sum().sum())

print(
    "Infinite values:",
    np.isinf(final_feature_matrix.to_numpy()).sum()
)

print("\nMinimum feature value:", final_feature_matrix.min().min())
print("Maximum feature value:", final_feature_matrix.max().max())

print(
    "\nConstant features:",
    (final_feature_matrix.nunique() <= 1).sum()
)

print(
    "Feature order matches saved feature list:",
    final_feature_matrix.columns.tolist() == feature_names
)

Final dataset shape: (7859, 81)
Unique patients: 7859
Duplicate patient IDs: 0

Number of clustering features: 80

Missing values: 0
Infinite values: 0

Minimum feature value: 0.0
Maximum feature value: 1.0000000000000002

Constant features: 0
Feature order matches saved feature list: True


#### Observation :

The final clustering dataset contains `7,859` unique patients and `80` clustering features.

There are no duplicate patients, missing values, infinite values, or constant features.

All clustering variables are scaled to approximately `0–1`, and the feature order matches the saved feature list.


## Save Final Processed Datasets and Feature List

In [39]:
# Save the combined dataset before scaling
feature_data_imputed.to_csv(
    RESULTS_DIR / "feature_data_imputed.csv",
    index=False
)

# Save the encoded dataset before scaling
feature_data_encoded.to_csv(
    RESULTS_DIR / "feature_data_encoded.csv",
    index=False
)

# Save the final scaled clustering dataset
scaled_features.to_csv(
    RESULTS_DIR / "clustering_dataset_scaled.csv",
    index=False
)

# Save the clustering feature names in the exact column order
pd.Series(
    feature_names,
    name="feature_name"
).to_csv(
    RESULTS_DIR / "clustering_feature_names.csv",
    index=False
)

# Save the combined vital-sign FPC features
vital_fpc_features.to_csv(
    RESULTS_DIR / "vital_fpc_features.csv",
    index=False
)

# Display saved outputs
print("Saved files:")

for file_name in [
    "feature_data_imputed.csv",
    "feature_data_encoded.csv",
    "clustering_dataset_scaled.csv",
    "clustering_feature_names.csv",
    "vital_fpc_features.csv"
]:
    print("-", RESULTS_DIR / file_name)

Saved files:
- C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation\feature_data_imputed.csv
- C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation\feature_data_encoded.csv
- C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation\clustering_dataset_scaled.csv
- C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation\clustering_feature_names.csv
- C:\Users\samsa\Documents\ICU Clustering\results\1.dataset_creation\vital_fpc_features.csv


## Create Final Cohort Flow Summary

In [40]:
cohort_flow = pd.DataFrame({
    "Step": [
        "Raw eICU patient table",
        "Sepsis ICU stays",
        "One sepsis ICU stay per patient",
        "Adult patients only",
        "After CSICU exclusion",
        "Complete required demographics",
        "All 12 selected labs available",
        "All 3 selected vital signs available",
        "Final dataset after height/weight imputation"
    ],
    "Patients / ICU stays": [
        200859,
        23136,
        20131,
        20119,
        19804,
        19617,
        8312,
        7859,
        7859
    ]
})

# Calculate how many records were removed at each step
cohort_flow["Removed from previous step"] = (
    cohort_flow["Patients / ICU stays"]
    .shift(1)
    - cohort_flow["Patients / ICU stays"]
)

# First row has no previous filtering step
cohort_flow.loc[0, "Removed from previous step"] = np.nan

print(cohort_flow.to_string(index=False))

                                        Step  Patients / ICU stays  Removed from previous step
                      Raw eICU patient table                200859                         NaN
                            Sepsis ICU stays                 23136                    177723.0
             One sepsis ICU stay per patient                 20131                      3005.0
                         Adult patients only                 20119                        12.0
                       After CSICU exclusion                 19804                       315.0
              Complete required demographics                 19617                       187.0
              All 12 selected labs available                  8312                     11305.0
        All 3 selected vital signs available                  7859                       453.0
Final dataset after height/weight imputation                  7859                         0.0


#### Observation :

The cohort flow clearly shows how the dataset was reduced from `200,859` raw ICU stays to the final `7,859` patients.

The largest reduction occurred when requiring all `12` selected laboratory variables, which removed `11,305` patients.

Height and weight imputation removed no patients, allowing the final cohort to remain at `7,859`.

## Final Cohort Flow With ICU Stays and Unique Patients Separately

In [41]:
cohort_flow = pd.DataFrame({
    "Step": [
        "Raw eICU patient table",
        "Sepsis filtering",
        "One sepsis ICU stay per patient",
        "Adult patients only",
        "After CSICU exclusion",
        "Complete required demographics",
        "All 12 selected labs available",
        "All 3 selected vital signs available",
        "Final dataset after height/weight imputation"
    ],

    "ICU stays": [
        200859,
        23136,
        20131,
        20119,
        19804,
        19617,
        8312,
        7859,
        7859
    ],

    "Unique patients": [
        139367,
        20131,
        20131,
        20119,
        19804,
        19617,
        8312,
        7859,
        7859
    ]
})

# Calculate ICU stays removed at each step
cohort_flow["ICU stays removed"] = (
    cohort_flow["ICU stays"].shift(1)
    - cohort_flow["ICU stays"]
)

# Calculate unique patients removed at each step
cohort_flow["Unique patients removed"] = (
    cohort_flow["Unique patients"].shift(1)
    - cohort_flow["Unique patients"]
)

# First row has no previous filtering step
cohort_flow.loc[0, ["ICU stays removed", "Unique patients removed"]] = np.nan

print(cohort_flow.to_string(index=False))

                                        Step  ICU stays  Unique patients  ICU stays removed  Unique patients removed
                      Raw eICU patient table     200859           139367                NaN                      NaN
                            Sepsis filtering      23136            20131           177723.0                 119236.0
             One sepsis ICU stay per patient      20131            20131             3005.0                      0.0
                         Adult patients only      20119            20119               12.0                     12.0
                       After CSICU exclusion      19804            19804              315.0                    315.0
              Complete required demographics      19617            19617              187.0                    187.0
              All 12 selected labs available       8312             8312            11305.0                  11305.0
        All 3 selected vital signs available       7859         